# MoE Experiment 3 — `moe_sym_bal08_v1` (Google Colab)

Symmetric router (no jitter/dropout), **4 experts**, top-k=1, fast strong balance.

**Before you run**
1. Runtime → **GPU** (T4 or better).
2. Kaggle API token for COCO + checkpoint download (`kaggle.json`).
3. Colab disk ~100 GB — full COCO needs ~25 GB extracted. Use **Google Drive** if space runs out.

**Epoch 10 gates:** val `expert_max_use` < 0.50, `train_val_load_l1` < 0.20

## 1. Configuration — edit these

In [ ]:
# --- Colab paths (everything under /content) ---
PROJECT_ROOT = "/content/newmethod"
MOE_DIR = f"{PROJECT_ROOT}/hidden_frozen178_moe"
DATA_DIR = "/content/coco100k"
CHECKPOINT_PATH = "/content/checkpoints/my_hidden_experiment--epoch-177.pyt"
EXTRACT_DIR = "/content/coco_extract"  # zip extract target (writable)

# --- git ---
REPO_URL = "https://github.com/ademladhari/newmethod.git"
REPO_BRANCH = "main"

# --- Kaggle API (COCO + checkpoint download) ---
KAGGLE_USERNAME = ""  # or upload kaggle.json in next section
KAGGLE_KEY = ""

# --- training ---
BATCH_SIZE = 16   # try 32 on A100 if no OOM
EPOCHS = 20
NUM_WORKERS = 2
EXPERIMENT_NAME = "moe_sym_bal08_v1"

# --- optional: skip download if COCO already on Drive ---
USE_GOOGLE_DRIVE = False
DRIVE_DATA_DIR = "/content/drive/MyDrive/coco100k"
DRIVE_CHECKPOINT = "/content/drive/MyDrive/checkpoints/my_hidden_experiment--epoch-177.pyt"

## 2. GPU check + optional Drive mount

In [ ]:
import os
import shutil
import subprocess

import torch

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)

subprocess.run(["nvidia-smi"], check=False)

## 3. Kaggle credentials

In [ ]:
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

# Option A: set KAGGLE_USERNAME / KAGGLE_KEY in the config cell above
if KAGGLE_USERNAME and KAGGLE_KEY:
    kaggle_json = kaggle_dir / "kaggle.json"
    kaggle_json.write_text(
        '{{"username":"{}","key":"{}"}}\n'.format(KAGGLE_USERNAME, KAGGLE_KEY)
    )
    os.chmod(kaggle_json, 0o600)
    print("Wrote", kaggle_json)
else:
    # Option B: upload kaggle.json manually
    try:
        from google.colab import files
        uploaded = files.upload()  # pick kaggle.json
        if "kaggle.json" in uploaded:
            (kaggle_dir / "kaggle.json").write_bytes(uploaded["kaggle.json"])
            os.chmod(kaggle_dir / "kaggle.json", 0o600)
            print("Uploaded kaggle.json")
    except Exception as e:
        print("Set KAGGLE_USERNAME/KAGGLE_KEY or upload kaggle.json:", e)

assert (kaggle_dir / "kaggle.json").is_file(), "Missing ~/.kaggle/kaggle.json"

## 4. Clone repo + install deps

In [ ]:
if os.path.isdir(PROJECT_ROOT):
    !rm -rf {PROJECT_ROOT}

!git clone --branch {REPO_BRANCH} {REPO_URL} {PROJECT_ROOT}
%cd {MOE_DIR}

# kagglehub 0.3.12 works on Linux; pin to avoid broken 1.x on some setups
!pip -q install kagglehub==0.3.12 kaggle

import sys
sys.path.insert(0, MOE_DIR)
print("Working dir:", os.getcwd())

## 5. Download COCO 2017 → `train/` and `val/`

In [ ]:
import zipfile
from pathlib import Path

train_dst = Path(DATA_DIR) / "train"
val_dst = Path(DATA_DIR) / "val"
marker = Path(DATA_DIR) / ".ready"


def count_jpg(folder: Path) -> int:
    if not (folder.is_dir() or folder.is_symlink()):
        return 0
    return sum(1 for f in folder.iterdir() if f.suffix.lower() == ".jpg")


def find_image_dirs(root: Path):
    train_dir = val_dir = None
    for dirpath, _, filenames in os.walk(root):
        if not filenames:
            continue
        base = Path(dirpath).name.lower()
        if base == "train2017" and any(f.lower().endswith(".jpg") for f in filenames):
            train_dir = Path(dirpath)
        if base in ("val2017", "valid2017") and any(f.lower().endswith(".jpg") for f in filenames):
            val_dir = Path(dirpath)
    return train_dir, val_dir


def setup_train_val_symlinks(train_src: Path, val_src: Path):
    """Never shutil.move — kagglehub cache can be read-only / another mount on Colab."""
    Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
    for dst in (train_dst, val_dst):
        if dst.is_symlink():
            dst.unlink()
        elif dst.is_dir():
            shutil.rmtree(dst)

    os.symlink(train_src.resolve(), train_dst, target_is_directory=True)
    os.symlink(val_src.resolve(), val_dst, target_is_directory=True)
    marker.write_text("ok\n")
    print("Linked train ->", train_src)
    print("Linked val   ->", val_src)


if USE_GOOGLE_DRIVE and Path(DRIVE_DATA_DIR).is_dir() and count_jpg(Path(DRIVE_DATA_DIR) / "train") > 1000:
    print("Using COCO from Google Drive:", DRIVE_DATA_DIR)
    setup_train_val_symlinks(Path(DRIVE_DATA_DIR) / "train", Path(DRIVE_DATA_DIR) / "val")
elif marker.is_file() and count_jpg(train_dst) > 100000 and count_jpg(val_dst) > 4000:
    print("COCO already prepared at", DATA_DIR)
else:
    print("Downloading COCO 2017 via kagglehub...")
    import kagglehub

    cache_path = Path(kagglehub.dataset_download("awsaf49/coco-2017-dataset"))
    print("kagglehub cache:", cache_path)

    train_src, val_src = find_image_dirs(cache_path)

    if train_src is None or val_src is None:
        # Extract zip into writable /content (NOT move from cache)
        archive = cache_path / "2.archive"
        if not archive.is_file():
            for p in cache_path.rglob("*.archive"):
                archive = p
                break
        extract_root = Path(EXTRACT_DIR)
        train_marker = extract_root / ".extract_done"
        if not train_marker.is_file():
            extract_root.mkdir(parents=True, exist_ok=True)
            print("Extracting to", extract_root, "(~15 min)...")
            with zipfile.ZipFile(archive, "r") as zf:
                zf.extractall(extract_root)
            train_marker.write_text("ok\n")
        train_src, val_src = find_image_dirs(extract_root)

    assert train_src and val_src, "Could not find train2017/val2017"
    setup_train_val_symlinks(train_src, val_src)

print("train images:", count_jpg(train_dst))
print("val images:", count_jpg(val_dst))
print("data-dir:", DATA_DIR)

## 6. Download HiDDeN epoch-177 checkpoint

In [ ]:
ckpt = Path(CHECKPOINT_PATH)
ckpt.parent.mkdir(parents=True, exist_ok=True)

if USE_GOOGLE_DRIVE and Path(DRIVE_CHECKPOINT).is_file():
    print("Copying checkpoint from Drive...")
    shutil.copy2(DRIVE_CHECKPOINT, ckpt)
elif ckpt.is_file() and ckpt.stat().st_size > 5_000_000:
    print("Checkpoint already present:", ckpt)
else:
    print("Downloading checkpoint from Kaggle dataset...")
    import kagglehub

    ckpt_cache = kagglehub.dataset_download("wings2ofice2/hiddencheckpointc")
    matches = list(Path(ckpt_cache).rglob("*epoch-177.pyt"))
    if not matches:
        matches = list(Path(ckpt_cache).rglob("*.pyt"))
    assert matches, "No .pyt checkpoint found in Kaggle dataset"
    src = max(matches, key=lambda p: p.stat().st_size)
    shutil.copy2(src, ckpt)
    print("Saved:", ckpt, "size MB:", round(ckpt.stat().st_size / 1e6, 2))

assert ckpt.is_file(), "Checkpoint missing — upload to Drive or fix Kaggle dataset slug"

## 7. Experiment 3 — train

In [ ]:
import sys

os.chdir(MOE_DIR)
sys.path.insert(0, MOE_DIR)

train_cmd = [
    sys.executable, "train_moe.py", "new",
    "--data-dir", DATA_DIR,
    "--name", EXPERIMENT_NAME,
    "--batch-size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--num-experts", "4",
    "--top-k", "1",
    "--balance-loss-weight", "0.08",
    "--balance-loss-start-weight", "0.02",
    "--balance-loss-warmup-epochs", "5",
    "--router-jitter-noise", "0.0",
    "--router-input-dropout", "0.0",
    "--expert-dropout", "0.0",
    "--router-z-loss-weight", "0.001",
    "--router-temperature-start", "1.4",
    "--router-temperature-end", "1.0",
    "--load-penalty-weight", "0.0",
    "--adversarial-loss", "0.0001",
    "--init-hidden-checkpoint", CHECKPOINT_PATH,
    "--freeze-hidden-backbone",
    "--enable-fp16",
    "--router-grad-clip-norm", "0.5",
    "--num-workers", str(NUM_WORKERS),
    "--save-every", "5",
    "--print-each", "100",
]
print(" ".join(train_cmd))
subprocess.run(train_cmd, check=True, cwd=MOE_DIR)

## 8. (Optional) Save run to Google Drive

In [ ]:
from glob import glob

run_dirs = sorted(glob(f"{MOE_DIR}/runs/{EXPERIMENT_NAME}*"))
if not run_dirs:
    print("No run folder found under runs/")
else:
    latest = run_dirs[-1]
    print("Latest run:", latest)
    for name in ["train.csv", "validation.csv", "validation_noisy.csv"]:
        p = os.path.join(latest, name)
        if os.path.isfile(p):
            print(name, "— last epoch row preview:")
            subprocess.run(["tail", "-n", "2", p], check=False)

    run_name = os.path.basename(latest)
    zip_path = f"/content/{EXPERIMENT_NAME}_run.zip"

    if USE_GOOGLE_DRIVE:
        dest = f"/content/drive/MyDrive/moe_runs/{EXPERIMENT_NAME}"
        os.makedirs(dest, exist_ok=True)
        subprocess.run(["cp", "-r", latest, dest], check=True)
        print("Copied to", dest)
    else:
        print("Zip and download locally:")
        subprocess.run(
            ["zip", "-r", zip_path, run_name],
            check=True,
            cwd=f"{MOE_DIR}/runs",
        )
        try:
            from google.colab import files
            files.download(zip_path)
        except Exception:
            print("Download from Colab file browser:", zip_path)